[Apache DataFusion](https://datafusion.apache.org/) is an extensible query engine written in Rust that uses [Apache Arrow](https://arrow.apache.org/) as its in-memory format. One of its powerful features is table providers—an abstraction that allows DataFusion to query data from virtually any source while presenting a unified SQL interface. The [datafusion-table-providers](https://github.com/datafusion-contrib/datafusion-table-providers) crate offers ready-made table provider implementations for common data sources, including databases accessible via ADBC.

[ADBC](https://arrow.apache.org/adbc/) (Arrow Database Connectivity) is a database-agnostic API that retrieves query results directly in Arrow format, making it an efficient bridge between DataFusion and external databases. This demo uses DuckDB for simplicity, but ADBC's real value is providing DataFusion with access to more than a dozen different data systems that lack dedicated table providers.

In this notebook, you will:

1. Load an ADBC driver (DuckDB) and connect to a database.
2. Create an ADBC connection pool and table factory.
3. Register an external database table as a DataFusion table provider.
4. Query the data through DataFusion's SQL interface.

Install the DuckDB ADBC driver with [uv](https://docs.astral.sh/uv/) and [dbc](https://docs.columnar.tech/dbc/):

In [ ]:
!uvx dbc install -q duckdb

Load the required crates:

In [2]:
:dep adbc_core = { version = "0.21" }
:dep adbc_driver_manager = { version = "0.21" }
:dep datafusion = { version = "52.0", default-features = false, features = ["sql"] }
:dep datafusion-table-providers = { version = "0.10.1", features = ["adbc"] }

Import the necessary types:

In [3]:
use adbc_core::options::{AdbcVersion, OptionDatabase};
use adbc_core::{Driver, LOAD_FLAG_DEFAULT};
use adbc_driver_manager::ManagedDriver;
use datafusion::prelude::SessionContext;
use datafusion::sql::TableReference;
use datafusion_table_providers::adbc::AdbcTableFactory;
use datafusion_table_providers::sql::db_connection_pool::adbcpool::ADBCPool;
use std::sync::Arc;

Load the DuckDB ADBC driver and create a database handle:

In [4]:
let mut driver = ManagedDriver::load_from_name(
    "duckdb",
    None,
    AdbcVersion::default(),
    LOAD_FLAG_DEFAULT,
    None,
)
.unwrap();

let database = driver
    .new_database_with_opts([(OptionDatabase::Uri, "../data/penguins.duckdb".into())])
    .unwrap();

Create an ADBC connection pool and table factory:

In [5]:
let adbc_pool = Arc::new(ADBCPool::new(database, None).unwrap());
let table_factory = AdbcTableFactory::new(adbc_pool.clone());

Create a DataFusion session context:

In [6]:
let ctx = SessionContext::new();

Register the `penguins` table from DuckDB as a DataFusion table using the ADBC table factory:

In [7]:
ctx.register_table(
    "penguins",
    table_factory
        .table_provider(TableReference::bare("penguins"), None)
        .await
        .unwrap(),
)
.unwrap()

None

Execute a SQL query through DataFusion (which fetches data from DuckDB via ADBC) and display the results:

In [9]:
let df = ctx
    .sql("SELECT * FROM datafusion.public.penguins LIMIT 10")
    .await
    .unwrap();

df.show().await.unwrap();

+---------+-----------+----------------+---------------+-------------------+-------------+--------+------+
| species | island    | bill_length_mm | bill_depth_mm | flipper_length_mm | body_mass_g | sex    | year |
+---------+-----------+----------------+---------------+-------------------+-------------+--------+------+
| Adelie  | Torgersen | 39.1           | 18.7          | 181               | 3750        | male   | 2007 |
| Adelie  | Torgersen | 39.5           | 17.4          | 186               | 3800        | female | 2007 |
| Adelie  | Torgersen | 40.3           | 18.0          | 195               | 3250        | female | 2007 |
| Adelie  | Torgersen |                |               |                   |             |        | 2007 |
| Adelie  | Torgersen | 36.7           | 19.3          | 193               | 3450        | female | 2007 |
| Adelie  | Torgersen | 39.3           | 20.6          | 190               | 3650        | male   | 2007 |
| Adelie  | Torgersen | 38.9         